# Extract Subject-Level Acoustic Tracking Features

Use this notebook after TRF outputs have been generated. It reads TRF-Tools outputs and summarizes tracking into subject-level feature columns.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_trf_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_trf_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_trf_experiment import BIDS_ROOT, PARAMETERS, alice

MODEL = 'gammatone-8'
FEATURE_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/features')
QC_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/qc')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

subjects = alice.get_field_values('subject')
print(f'Subjects: {len(subjects)}')
print(f'Model: {MODEL}')

INFO    :  *** AliceComprehensionTRF initialized with root /Users/yanyuwoo/Data/bids on 2026-07-08 17:08:25 ***
INFO    :  Using eelbrain 0.42.0a4, mne 1.11.0.
Subjects: 49
Model: gammatone-8


## Check Which TRF Outputs Exist

This does not fit any model. It only checks cache paths.

In [3]:
cache_rows = []
for subject in subjects:
    # make=True is needed for first-time model-name registration; path_only=True prevents fitting.
    path = Path(alice.load_trf(MODEL, subject=subject, make=True, path_only=True, **PARAMETERS))
    cache_rows.append({'subject': f'sub-{subject}', 'trf_path': str(path), 'exists': path.exists()})

cache_report = pd.DataFrame(cache_rows)
display(cache_report['exists'].value_counts(dropna=False).rename('n_subjects'))
display(cache_report.head())
cache_report.to_csv(QC_DIR / 'trf_gammatone8_cache_status.csv', index=False)

exists
False    49
Name: n_subjects, dtype: int64

,subject,trf_path,exists
0,sub-01,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
1,sub-02,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
2,sub-03,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
3,sub-04,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
4,sub-05,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False


## Inspect One Available Output

Run this before aggregating. It shows which metrics TRF-Tools produced for the first available subject.

In [4]:
available = cache_report.loc[cache_report['exists'], 'subject'].str.replace('sub-', '', regex=False).tolist()
if not available:
    print('No cached TRF outputs found yet. Run notebook 02 for one subject or notebook 03 for batch first.')
else:
    example_subject = available[0]
    ds = alice.load_trfs(example_subject, MODEL, make=False, **PARAMETERS)
    print(f'Example subject: sub-{example_subject}')
    print(ds)
    print('\nDataset keys:')
    print(list(ds.keys()))

No cached TRF outputs found yet. Run notebook 02 for one subject or notebook 03 for batch first.


## Aggregate Tracking Scores

`tracking_r_mean` is the main subject-level feature. It averages the TRF validation correlation across available `r` values. `tracking_r_best` is diagnostic only because max scores are more sensitive to leakage and noisy channels.

In [5]:
def numeric_values(value):
    arr = np.asarray(value, dtype=float)
    return arr[np.isfinite(arr)]


feature_rows = []
failed_rows = []

for subject in subjects:
    subject_label = f'sub-{subject}'
    try:
        path = Path(alice.load_trf(MODEL, subject=subject, make=True, path_only=True, **PARAMETERS))
        if not path.exists():
            failed_rows.append({'subject': subject_label, 'reason': 'missing_trf_cache', 'path': str(path)})
            continue

        ds = alice.load_trfs(subject, MODEL, make=False, **PARAMETERS)
        if 'r' not in ds:
            failed_rows.append({'subject': subject_label, 'reason': 'dataset_has_no_r', 'path': str(path)})
            continue

        r_values = numeric_values(ds['r'])
        if len(r_values) == 0:
            failed_rows.append({'subject': subject_label, 'reason': 'empty_r_values', 'path': str(path)})
            continue

        feature_rows.append({
            'subject': subject_label,
            'predictor_name': MODEL,
            'tracking_r_mean': float(np.mean(r_values)),
            'tracking_r_median': float(np.median(r_values)),
            'tracking_r_best': float(np.max(r_values)),
            'tracking_r_min': float(np.min(r_values)),
            'n_r_values': int(len(r_values)),
            'trf_path': str(path),
            'status': 'ok',
        })
    except Exception as exc:
        failed_rows.append({'subject': subject_label, 'reason': repr(exc), 'path': ''})

features = pd.DataFrame(feature_rows)
qc = pd.DataFrame(failed_rows)

feature_path = FEATURE_DIR / 'trf_gammatone8_acoustic_tracking.csv'
qc_path = QC_DIR / 'trf_gammatone8_feature_extraction_qc.csv'
features.to_csv(feature_path, index=False)
qc.to_csv(qc_path, index=False)

print(f'Wrote features: {feature_path}')
print(f'Wrote QC: {qc_path}')
print(f'Feature rows: {len(features)}')
print(f'QC rows: {len(qc)}')
display(features.head())
display(qc.head())

Wrote features: /Users/yanyuwoo/Data/Alice Comprehension/features/trf_gammatone8_acoustic_tracking.csv
Wrote QC: /Users/yanyuwoo/Data/Alice Comprehension/qc/trf_gammatone8_feature_extraction_qc.csv
Feature rows: 0
QC rows: 49


""


,subject,reason,path
0,sub-01,missing_trf_cache,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
1,sub-02,missing_trf_cache,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
2,sub-03,missing_trf_cache,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
3,sub-04,missing_trf_cache,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
4,sub-05,missing_trf_cache,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
